# 📊 Module 4.2: Dynamic Programming for Optimal Image Processing

## From Bellman Equations to Optimal Image Pipelines

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_04_Basic_RL_Algorithms/02_Dynamic_Programming/dynamic_programming.ipynb)

---

## 🎯 Learning Objectives
1. Derive the **Bellman expectation** and **Bellman optimality** equations from first principles
2. Implement **iterative policy evaluation** and prove convergence via contraction mapping
3. Implement **policy iteration** and **value iteration** on grid MDPs
4. Apply DP to find the optimal sequence of image processing operations

**Prerequisites:** MDP formulation (Module 3), basic linear algebra

---

In [ ]:
# Setup - Install dependencies (Google Colab)
!pip install numpy matplotlib opencv-python-headless pillow scikit-image -q

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
print("✅ All libraries loaded successfully!")

In [ ]:
# Download REAL open-source dataset
import torchvision
import numpy as np

# CIFAR-10 for image filter selection environment
cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
# Take small subset for fast RL experiments
real_images = [np.array(cifar10[i][0]) for i in range(200)]
print(f"✅ CIFAR-10 loaded: Using {len(real_images)} real photos for RL environment")
print(f"   Image shape: {real_images[0].shape}")
print("   These will be our 'states' that the RL agent processes!")

## Deep Derivation: Dynamic Programming Convergence and Complexity

### Step 1: Iterative Policy Evaluation Update Rule
$$V_{k+1}(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a)\left[R(s,a,s') + \gamma V_k(s')\right]$$

In operator notation: $\mathbf{v}_{k+1} = \mathcal{T}^\pi \mathbf{v}_k$ where $\mathcal{T}^\pi$ is the Bellman evaluation operator.

### Step 2: Proof That Policy Evaluation Converges

**Theorem:** $\mathcal{T}^\pi$ is a $\gamma$-contraction in the $\|\cdot\|_\infty$ norm.

**Proof:**
$$\|\mathcal{T}^\pi V_1 - \mathcal{T}^\pi V_2\|_\infty = \max_s \left|\sum_a \pi(a|s)\sum_{s'} P(s'|s,a)\gamma[V_1(s') - V_2(s')]\right|$$
$$\leq \max_s \sum_a \pi(a|s) \sum_{s'} P(s'|s,a) \gamma |V_1(s') - V_2(s')|$$
$$\leq \gamma \|V_1 - V_2\|_\infty \cdot \max_s \underbrace{\sum_a \pi(a|s)}_{=1} \underbrace{\sum_{s'} P(s'|s,a)}_{=1} = \gamma \|V_1 - V_2\|_\infty$$

Since $\gamma < 1$, by Banach Fixed-Point Theorem, $V_k \to V^\pi$ at rate $O(\gamma^k)$. $\blacksquare$

### Step 3: Policy Evaluation — Matrix Solution (Direct Method)
The Bellman equation in matrix form: $\mathbf{v}^\pi = \mathbf{r}^\pi + \gamma\mathbf{P}^\pi\mathbf{v}^\pi$

**Derivation of closed-form solution:**
$$\mathbf{v}^\pi - \gamma\mathbf{P}^\pi\mathbf{v}^\pi = \mathbf{r}^\pi$$
$$(\mathbf{I} - \gamma\mathbf{P}^\pi)\mathbf{v}^\pi = \mathbf{r}^\pi$$
$$\mathbf{v}^\pi = (\mathbf{I} - \gamma\mathbf{P}^\pi)^{-1}\mathbf{r}^\pi$$

**Proof of invertibility:** Eigenvalues of $\gamma\mathbf{P}^\pi$ have magnitude $\leq \gamma < 1$, so all eigenvalues of $\mathbf{I} - \gamma\mathbf{P}^\pi$ are non-zero. $\blacksquare$

**Complexity comparison:**
- Direct solve: $O(|\mathcal{S}|^3)$ (matrix inversion)
- Iterative: $O(|\mathcal{S}|^2|\mathcal{A}|)$ per iteration, $\frac{\log(1/\epsilon)}{\log(1/\gamma)}$ iterations

### Step 4: Policy Improvement Step — Derivation
**Greedy policy:** $\pi'(s) = \arg\max_a Q^\pi(s, a) = \arg\max_a \sum_{s'} P(s'|s,a)[R(s,a,s') + \gamma V^\pi(s')]$

**Proof that $\pi'$ is at least as good as $\pi$:**
For all $s$: $Q^\pi(s, \pi'(s)) = \max_a Q^\pi(s,a) \geq Q^\pi(s, \pi(s)) = V^\pi(s)$

By the Policy Improvement Theorem (proved in Module 3.5): $V^{\pi'} \geq V^\pi$. $\blacksquare$

### Step 5: Policy Iteration — Finite Convergence
**Theorem:** Policy iteration converges in at most $|\mathcal{A}|^{|\mathcal{S}|}$ iterations.

**Proof:**
1. Each iteration strictly improves the policy (unless already optimal)
2. There are at most $|\mathcal{A}|^{|\mathcal{S}|}$ deterministic policies
3. We never revisit a policy (strict improvement at each non-terminal step)
4. Therefore convergence in finite steps $\blacksquare$

**In practice:** Policy iteration typically converges in very few iterations (often $< 10$), far less than the worst case.

### Step 6: Value Iteration — Combining Steps
$$V_{k+1}(s) = \max_a \sum_{s'} P(s'|s,a)[R(s,a,s') + \gamma V_k(s')]$$

**Proof that value iteration is equivalent to policy iteration with single-step evaluation:**
The max operation implicitly performs policy improvement. The expectation performs one step of policy evaluation. Combining both into a single update yields value iteration. $\blacksquare$

### Step 7: Stopping Criterion Derivation
**Theorem:** If $\|V_{k+1} - V_k\|_\infty < \frac{\epsilon(1-\gamma)}{2\gamma}$, then $\|V_k - V^*\|_\infty < \epsilon$.

**Proof:**
$$\|V_k - V^*\|_\infty \leq \frac{\gamma}{1-\gamma}\|V_k - V_{k-1}\|_\infty < \frac{\gamma}{1-\gamma} \cdot \frac{\epsilon(1-\gamma)}{2\gamma} = \frac{\epsilon}{2} < \epsilon \quad \blacksquare$$

### RL Connection: DP for Optimal Image Processing Pipelines
DP finds the optimal sequence of operations when the MDP model is known:
- States: discretized image quality levels
- Actions: {sharpen, blur, denoise, enhance contrast, adjust brightness}
- Known transitions: deterministic effect of each filter
- DP computes the optimal pipeline for any starting image quality

## 1. The Dynamic Programming Principle

### 1.1 Bellman's Principle of Optimality

> *An optimal policy has the property that whatever the initial state and decision are, the remaining decisions must constitute an optimal policy with regard to the state resulting from the first decision.* — Richard Bellman, 1957

### 1.2 Requirements for DP

DP methods require:
1. A **perfect model** of the environment (complete knowledge of $p(s', r | s, a)$)
2. A **finite MDP** ($|\mathcal{S}|, |\mathcal{A}|$ are finite)

### 1.3 Computational Complexity

- Policy evaluation: $O(|\mathcal{S}|^2 |\mathcal{A}|)$ per iteration
- Policy iteration: polynomial in $|\mathcal{S}|$ and $|\mathcal{A}|$
- Value iteration: $O(|\mathcal{S}|^2 |\mathcal{A}|)$ per iteration

Despite being computationally intensive, DP provides **exact solutions** and is the foundation for understanding all RL algorithms.

## 2. Bellman Equations — The Foundation

### 2.1 State-Value Function

The **state-value function** under policy $\pi$:

$$v_\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s] = \mathbb{E}_\pi\left[\sum_{k=0}^{\infty} \gamma^k R_{t+k+1} \mid S_t = s\right]$$

### 2.2 Bellman Expectation Equation (Derivation)

Starting from the definition:

$$v_\pi(s) = \mathbb{E}_\pi[R_{t+1} + \gamma G_{t+1} \mid S_t = s]$$

Expanding by conditioning on action and next state:

$$v_\pi(s) = \sum_{a} \pi(a|s) \sum_{s'} \sum_{r} p(s', r | s, a) \left[r + \gamma v_\pi(s')\right]$$

This is the **Bellman expectation equation** — it expresses $v_\pi(s)$ recursively in terms of $v_\pi(s')$.

### 2.3 Action-Value Bellman Equation

$$q_\pi(s, a) = \sum_{s', r} p(s', r | s, a)\left[r + \gamma \sum_{a'} \pi(a'|s') q_\pi(s', a')\right]$$

### 2.4 Bellman Optimality Equations

For the **optimal** value functions $v_*$ and $q_*$:

$$v_*(s) = \max_a \sum_{s', r} p(s', r | s, a)\left[r + \gamma v_*(s')\right]$$

$$q_*(s, a) = \sum_{s', r} p(s', r | s, a)\left[r + \gamma \max_{a'} q_*(s', a')\right]$$

### 2.5 Matrix Form

For a fixed policy $\pi$, define:
- $\mathbf{v}_\pi \in \mathbb{R}^{|\mathcal{S}|}$: vector of state values
- $\mathbf{r}_\pi \in \mathbb{R}^{|\mathcal{S}|}$: expected immediate rewards under $\pi$
- $\mathbf{P}_\pi \in \mathbb{R}^{|\mathcal{S}| \times |\mathcal{S}|}$: transition matrix under $\pi$

The Bellman equation in matrix form:

$$\mathbf{v}_\pi = \mathbf{r}_\pi + \gamma \mathbf{P}_\pi \mathbf{v}_\pi$$

**Closed-form solution:**

$$\mathbf{v}_\pi = (\mathbf{I} - \gamma \mathbf{P}_\pi)^{-1} \mathbf{r}_\pi$$

This has complexity $O(|\mathcal{S}|^3)$, so iterative methods are preferred for large state spaces.

In [ ]:
class GridWorldMDP:
    """A simple grid world MDP for demonstrating DP algorithms."""
    
    def __init__(self, size=4):
        self.size = size
        self.n_states = size * size
        self.n_actions = 4  # up, right, down, left
        self.action_names = ['↑', '→', '↓', '←']
        self.action_deltas = [(-1, 0), (0, 1), (1, 0), (0, -1)]
        
        # Terminal states: top-left and bottom-right corners
        self.terminal_states = {0, self.n_states - 1}
        
        # Build transition model p(s', r | s, a)
        self._build_model()
    
    def state_to_rc(self, s):
        return s // self.size, s % self.size
    
    def rc_to_state(self, r, c):
        return r * self.size + c
    
    def _build_model(self):
        """Build transition probabilities and rewards."""
        S, A = self.n_states, self.n_actions
        
        # P[s, a, s'] = probability of transitioning to s' from s with action a
        self.P = np.zeros((S, A, S))
        # R[s, a] = expected reward for taking action a in state s
        self.R = np.zeros((S, A))
        
        for s in range(S):
            if s in self.terminal_states:
                for a in range(A):
                    self.P[s, a, s] = 1.0
                    self.R[s, a] = 0.0
                continue
            
            r, c = self.state_to_rc(s)
            for a in range(A):
                dr, dc = self.action_deltas[a]
                nr, nc = r + dr, c + dc
                
                if 0 <= nr < self.size and 0 <= nc < self.size:
                    s_next = self.rc_to_state(nr, nc)
                else:
                    s_next = s  # Bump into wall, stay
                
                self.P[s, a, s_next] = 1.0
                self.R[s, a] = -1.0  # Step cost

mdp = GridWorldMDP(size=4)
print(f"Grid World: {mdp.size}×{mdp.size} = {mdp.n_states} states, {mdp.n_actions} actions")
print(f"Terminal states: {mdp.terminal_states}")
print("Reward per step: -1 (encourages reaching terminal quickly)")

## 3. Policy Evaluation (Prediction)

### 3.1 Problem

Given a policy $\pi$, compute $v_\pi(s)$ for all $s \in \mathcal{S}$.

### 3.2 Iterative Policy Evaluation

Apply the Bellman expectation equation as an **update rule**:

$$v_{k+1}(s) = \sum_a \pi(a|s) \sum_{s', r} p(s', r | s, a)\left[r + \gamma v_k(s')\right]$$

starting from an arbitrary $v_0$.

### 3.3 Convergence via Contraction Mapping

Define the **Bellman operator** $T_\pi$:

$$(T_\pi v)(s) = \sum_a \pi(a|s) \sum_{s', r} p(s', r | s, a)[r + \gamma v(s')]$$

**Theorem:** $T_\pi$ is a $\gamma$-contraction in the $\ell_\infty$ norm:

$$\|T_\pi v - T_\pi u\|_\infty \leq \gamma \|v - u\|_\infty$$

**Proof sketch:**

$$|(T_\pi v)(s) - (T_\pi u)(s)| = \left|\sum_a \pi(a|s) \sum_{s'} p(s'|s,a) \gamma [v(s') - u(s')]\right|$$

$$\leq \gamma \sum_a \pi(a|s) \sum_{s'} p(s'|s,a) |v(s') - u(s')| \leq \gamma \|v - u\|_\infty$$

By the **Banach fixed-point theorem**, $T_\pi$ has a unique fixed point $v_\pi$, and $v_k \to v_\pi$ as $k \to \infty$.

**Convergence rate:** $\|v_k - v_\pi\|_\infty \leq \gamma^k \|v_0 - v_\pi\|_\infty$

### 3.4 Algorithm

$$\boxed{\textbf{Iterative Policy Evaluation}}$$

**Input:** Policy $\pi$, threshold $\theta > 0$

1. Initialize $v(s) = 0$ for all $s \in \mathcal{S}$
2. **Repeat:**
   - $\Delta \leftarrow 0$
   - For each $s \in \mathcal{S}$:
     - $v_{\text{old}} \leftarrow v(s)$
     - $v(s) \leftarrow \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a)[r + \gamma v(s')]$
     - $\Delta \leftarrow \max(\Delta, |v_{\text{old}} - v(s)|)$
3. **Until** $\Delta < \theta$

**Output:** $v \approx v_\pi$

In [ ]:
def iterative_policy_evaluation(mdp, policy, gamma=1.0, theta=1e-6, max_iters=1000):
    """Evaluate a policy using iterative Bellman updates.
    
    policy: array of shape (n_states, n_actions) — pi(a|s)
    Returns: value function and history of value snapshots.
    """
    V = np.zeros(mdp.n_states)
    history = [V.copy()]
    deltas = []
    
    for iteration in range(max_iters):
        delta = 0
        V_new = np.zeros_like(V)
        
        for s in range(mdp.n_states):
            if s in mdp.terminal_states:
                V_new[s] = 0
                continue
            
            v_s = 0
            for a in range(mdp.n_actions):
                # Sum over next states
                for s_next in range(mdp.n_states):
                    v_s += policy[s, a] * mdp.P[s, a, s_next] * \
                           (mdp.R[s, a] + gamma * V[s_next])
            
            V_new[s] = v_s
            delta = max(delta, abs(V[s] - V_new[s]))
        
        V = V_new
        history.append(V.copy())
        deltas.append(delta)
        
        if delta < theta:
            break
    
    return V, history, deltas

# Uniform random policy: pi(a|s) = 1/4
uniform_policy = np.ones((mdp.n_states, mdp.n_actions)) / mdp.n_actions

V_pi, history, deltas = iterative_policy_evaluation(mdp, uniform_policy, gamma=1.0)

print(f"Converged in {len(deltas)} iterations")
print("\nValue function under random policy:")
print(V_pi.reshape(mdp.size, mdp.size).round(1))

In [ ]:
def plot_value_function(V, mdp, title='Value Function', ax=None):
    """Plot value function as a heatmap on the grid."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    
    grid = V.reshape(mdp.size, mdp.size)
    im = ax.imshow(grid, cmap='RdYlGn', interpolation='nearest')
    
    for i in range(mdp.size):
        for j in range(mdp.size):
            ax.text(j, i, f'{grid[i, j]:.1f}', ha='center', va='center',
                    fontsize=12, fontweight='bold',
                    color='white' if abs(grid[i, j]) > (grid.max() - grid.min()) * 0.6 else 'black')
    
    ax.set_title(title)
    ax.set_xticks(range(mdp.size))
    ax.set_yticks(range(mdp.size))
    plt.colorbar(im, ax=ax, shrink=0.8)
    return ax

# Show value function evolution
snapshots = [0, 1, 3, 10, 50, len(history) - 1]
snapshots = [s for s in snapshots if s < len(history)]

fig, axes = plt.subplots(1, len(snapshots), figsize=(4 * len(snapshots), 4))
if len(snapshots) == 1:
    axes = [axes]

for idx, snap in enumerate(snapshots):
    plot_value_function(history[snap], mdp, f'Iteration {snap}', axes[idx])

plt.suptitle('Policy Evaluation: Value Function Evolution (Random Policy)', fontsize=14)
plt.tight_layout()
plt.show()

# Convergence plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(deltas, 'b-', linewidth=2)
ax.set_xlabel('Iteration')
ax.set_ylabel('Max |ΔV| (log scale)')
ax.set_title('Policy Evaluation Convergence')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Policy Improvement

### 4.1 Policy Improvement Theorem

**Theorem:** Let $\pi$ and $\pi'$ be two policies such that for all $s \in \mathcal{S}$:

$$q_\pi(s, \pi'(s)) \geq v_\pi(s)$$

Then $\pi'$ is at least as good as $\pi$:

$$v_{\pi'}(s) \geq v_\pi(s) \quad \forall s \in \mathcal{S}$$

**Proof:**

$$v_\pi(s) \leq q_\pi(s, \pi'(s)) = \mathbb{E}[R_{t+1} + \gamma v_\pi(S_{t+1}) \mid S_t=s, A_t=\pi'(s)]$$

$$\leq \mathbb{E}[R_{t+1} + \gamma q_\pi(S_{t+1}, \pi'(S_{t+1})) \mid S_t=s, A_t=\pi'(s)]$$

$$\leq \mathbb{E}[R_{t+1} + \gamma R_{t+2} + \gamma^2 q_\pi(S_{t+2}, \pi'(S_{t+2})) \mid \ldots]$$

$$\leq \ldots \leq \mathbb{E}_{\pi'}[G_t \mid S_t = s] = v_{\pi'}(s) \quad \blacksquare$$

### 4.2 Greedy Policy Construction

Given $v_\pi$, construct the **greedy** policy:

$$\pi'(s) = \arg\max_a q_\pi(s, a) = \arg\max_a \sum_{s', r} p(s', r | s, a)[r + \gamma v_\pi(s')]$$

By the policy improvement theorem, $v_{\pi'} \geq v_\pi$. If $v_{\pi'} = v_\pi$, then $\pi$ is already optimal.

In [ ]:
def compute_greedy_policy(mdp, V, gamma=1.0):
    """Compute the greedy policy w.r.t. value function V."""
    policy = np.zeros((mdp.n_states, mdp.n_actions))
    
    for s in range(mdp.n_states):
        if s in mdp.terminal_states:
            policy[s] = 1.0 / mdp.n_actions
            continue
        
        q_values = np.zeros(mdp.n_actions)
        for a in range(mdp.n_actions):
            for s_next in range(mdp.n_states):
                q_values[a] += mdp.P[s, a, s_next] * (mdp.R[s, a] + gamma * V[s_next])
        
        best_actions = np.where(q_values == q_values.max())[0]
        policy[s, best_actions] = 1.0 / len(best_actions)
    
    return policy

def plot_policy(policy, mdp, title='Policy', ax=None):
    """Plot policy as arrows on the grid."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    
    arrow_dx = [0, 0.3, 0, -0.3]   # up, right, down, left
    arrow_dy = [-0.3, 0, 0.3, 0]
    
    ax.set_xlim(-0.5, mdp.size - 0.5)
    ax.set_ylim(mdp.size - 0.5, -0.5)
    
    for s in range(mdp.n_states):
        r, c = mdp.state_to_rc(s)
        
        if s in mdp.terminal_states:
            ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                       facecolor='lightgreen', edgecolor='black'))
            ax.text(c, r, 'GOAL', ha='center', va='center', fontweight='bold', fontsize=10)
            continue
        
        ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                   facecolor='lightyellow', edgecolor='black'))
        
        for a in range(mdp.n_actions):
            if policy[s, a] > 0.01:
                ax.arrow(c, r, arrow_dx[a] * policy[s, a] * 2,
                         arrow_dy[a] * policy[s, a] * 2,
                         head_width=0.1, head_length=0.05,
                         fc='darkblue', ec='darkblue', alpha=0.8)
    
    ax.set_title(title)
    ax.set_xticks(range(mdp.size))
    ax.set_yticks(range(mdp.size))
    ax.grid(True, alpha=0.3)
    return ax

# Show improvement from random to greedy
greedy_policy = compute_greedy_policy(mdp, V_pi, gamma=1.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_policy(uniform_policy, mdp, 'Random Policy', axes[0])
plot_policy(greedy_policy, mdp, 'Greedy Policy (after evaluation)', axes[1])
plt.suptitle('Policy Improvement: Random → Greedy', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Policy Iteration

### 5.1 Algorithm

Policy iteration alternates between **evaluation** and **improvement**:

$$\pi_0 \xrightarrow{E} v_{\pi_0} \xrightarrow{I} \pi_1 \xrightarrow{E} v_{\pi_1} \xrightarrow{I} \pi_2 \xrightarrow{E} \cdots \xrightarrow{I} \pi_* \xrightarrow{E} v_*$$

$$\boxed{\textbf{Policy Iteration}}$$

1. Initialize $\pi$ arbitrarily
2. **Repeat:**
   - **Policy Evaluation:** Compute $v_\pi$ using iterative evaluation
   - **Policy Improvement:** $\pi'(s) = \arg\max_a \sum_{s',r} p(s',r|s,a)[r + \gamma v_\pi(s')]$
   - If $\pi' = \pi$: **stop** (optimal policy found)
   - Else: $\pi \leftarrow \pi'$

### 5.2 Convergence

Since there are finitely many deterministic policies ($|\mathcal{A}|^{|\mathcal{S}|}$) and each improvement step strictly increases $v_\pi$ (unless already optimal), policy iteration converges in a **finite number of steps**.

In [ ]:
def policy_iteration(mdp, gamma=1.0, theta=1e-6):
    """Full policy iteration algorithm."""
    # Initialize with random policy
    policy = np.ones((mdp.n_states, mdp.n_actions)) / mdp.n_actions
    
    V_history = []
    policy_history = []
    
    for iteration in range(100):
        # Policy Evaluation
        V, _, _ = iterative_policy_evaluation(mdp, policy, gamma, theta)
        V_history.append(V.copy())
        policy_history.append(policy.copy())
        
        # Policy Improvement
        new_policy = compute_greedy_policy(mdp, V, gamma)
        
        # Check convergence
        if np.allclose(policy, new_policy):
            print(f"Policy Iteration converged in {iteration + 1} iterations")
            break
        
        policy = new_policy
    
    return V, policy, V_history, policy_history

V_star_pi, pi_star_pi, V_hist_pi, pol_hist_pi = policy_iteration(mdp, gamma=1.0)

# Show policy iteration steps
n_steps = min(len(V_hist_pi), 4)
fig, axes = plt.subplots(2, n_steps, figsize=(5 * n_steps, 10))

for i in range(n_steps):
    plot_value_function(V_hist_pi[i], mdp, f'V (iter {i})', axes[0, i])
    plot_policy(pol_hist_pi[i], mdp, f'π (iter {i})', axes[1, i])

plt.suptitle('Policy Iteration: Value Functions and Policies', fontsize=14)
plt.tight_layout()
plt.show()

print("\nOptimal value function:")
print(V_star_pi.reshape(mdp.size, mdp.size).round(1))

## 6. Value Iteration

### 6.1 Derivation

Value iteration combines evaluation and improvement into a **single update**. Starting from the Bellman optimality equation:

$$v_*(s) = \max_a \sum_{s', r} p(s', r | s, a)[r + \gamma v_*(s')]$$

We turn this into an **iterative update**:

$$v_{k+1}(s) = \max_a \sum_{s', r} p(s', r | s, a)[r + \gamma v_k(s')]$$

This is equivalent to performing **one sweep of policy evaluation** followed by policy improvement.

### 6.2 Convergence

The **Bellman optimality operator** $T_*$:

$$(T_* v)(s) = \max_a \sum_{s', r} p(s', r | s, a)[r + \gamma v(s')]$$

is also a $\gamma$-contraction:

$$\|T_* v - T_* u\|_\infty \leq \gamma \|v - u\|_\infty$$

Therefore $v_k \to v_*$ as $k \to \infty$.

### 6.3 Algorithm

$$\boxed{\textbf{Value Iteration}}$$

1. Initialize $v(s) = 0$ for all $s$
2. **Repeat:**
   - $\Delta \leftarrow 0$
   - For each $s$:
     - $v_{\text{old}} \leftarrow v(s)$
     - $v(s) \leftarrow \max_a \sum_{s',r} p(s',r|s,a)[r + \gamma v(s')]$
     - $\Delta \leftarrow \max(\Delta, |v_{\text{old}} - v(s)|)$
3. **Until** $\Delta < \theta$
4. Extract policy: $\pi(s) = \arg\max_a \sum_{s',r} p(s',r|s,a)[r + \gamma v(s')]$

### 6.4 Comparison with Policy Iteration

| Property | Policy Iteration | Value Iteration |
|----------|-----------------|----------------|
| Per iteration | Full policy evaluation + improvement | Single Bellman optimality update |
| Cost per iteration | Higher (many evaluation sweeps) | Lower (one sweep) |
| Number of iterations | Fewer | More |
| Convergence | Exact in finite steps | Asymptotic |

In [ ]:
def value_iteration(mdp, gamma=1.0, theta=1e-6, max_iters=1000):
    """Value iteration algorithm."""
    V = np.zeros(mdp.n_states)
    history = [V.copy()]
    deltas = []
    
    for iteration in range(max_iters):
        delta = 0
        V_new = np.zeros_like(V)
        
        for s in range(mdp.n_states):
            if s in mdp.terminal_states:
                V_new[s] = 0
                continue
            
            q_values = np.zeros(mdp.n_actions)
            for a in range(mdp.n_actions):
                for s_next in range(mdp.n_states):
                    q_values[a] += mdp.P[s, a, s_next] * \
                                   (mdp.R[s, a] + gamma * V[s_next])
            
            V_new[s] = np.max(q_values)
            delta = max(delta, abs(V[s] - V_new[s]))
        
        V = V_new
        history.append(V.copy())
        deltas.append(delta)
        
        if delta < theta:
            print(f"Value Iteration converged in {iteration + 1} iterations")
            break
    
    policy = compute_greedy_policy(mdp, V, gamma)
    return V, policy, history, deltas

V_star_vi, pi_star_vi, V_hist_vi, deltas_vi = value_iteration(mdp, gamma=1.0)

# Show evolution
snapshots_vi = [0, 1, 3, 10, len(V_hist_vi) - 1]
snapshots_vi = [s for s in snapshots_vi if s < len(V_hist_vi)]

fig, axes = plt.subplots(1, len(snapshots_vi), figsize=(4 * len(snapshots_vi), 4))
for idx, snap in enumerate(snapshots_vi):
    plot_value_function(V_hist_vi[snap], mdp, f'Iteration {snap}', axes[idx])

plt.suptitle('Value Iteration: Value Function Evolution', fontsize=14)
plt.tight_layout()
plt.show()

# Final optimal policy
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_value_function(V_star_vi, mdp, 'Optimal Value Function V*', axes[0])
plot_policy(pi_star_vi, mdp, 'Optimal Policy π*', axes[1])
plt.tight_layout()
plt.show()

In [ ]:
# Convergence comparison: Policy Iteration vs Value Iteration
# Re-run policy iteration tracking per-sweep deltas
def policy_iteration_detailed(mdp, gamma=1.0, theta=1e-6):
    policy = np.ones((mdp.n_states, mdp.n_actions)) / mdp.n_actions
    all_deltas = []
    
    for iteration in range(100):
        V, _, eval_deltas = iterative_policy_evaluation(mdp, policy, gamma, theta)
        all_deltas.extend(eval_deltas)
        new_policy = compute_greedy_policy(mdp, V, gamma)
        if np.allclose(policy, new_policy):
            break
        policy = new_policy
    
    return all_deltas

pi_deltas = policy_iteration_detailed(mdp)

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(deltas_vi, 'b-', linewidth=2, label='Value Iteration', alpha=0.8)
ax.semilogy(pi_deltas, 'r-', linewidth=2, label='Policy Iteration (all sweeps)', alpha=0.8)
ax.set_xlabel('Total Bellman Sweeps')
ax.set_ylabel('Max |ΔV| (log scale)')
ax.set_title('Convergence Comparison: Policy Iteration vs Value Iteration')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Application: Optimal Image Processing Pipeline

### The MDP Formulation

- **States**: Discretized image quality levels (e.g., 5 levels each for contrast, sharpness, noise)
- **Actions**: Image operations (sharpen, denoise, contrast adjust, histogram equalize, edge enhance)
- **Transitions**: How each operation changes the quality levels
- **Rewards**: Improvement in overall quality
- **Goal**: Find the optimal sequence of operations to maximize image quality

In [ ]:
class ImageProcessingMDP:
    """MDP for sequential image processing.
    
    States encode (contrast_level, sharpness_level, noise_level)
    with n_levels for each dimension.
    """
    
    def __init__(self, n_levels=5):
        self.n_levels = n_levels
        self.n_states = n_levels ** 3 + 1  # +1 for terminal
        self.terminal_state = self.n_states - 1
        self.terminal_states = {self.terminal_state}
        
        self.action_names = ['Sharpen', 'Denoise', 'Contrast+', 'Hist Eq', 'Edge Enh', 'Done']
        self.n_actions = len(self.action_names)
        
        self._build_model()
    
    def state_to_features(self, s):
        if s == self.terminal_state:
            return None
        noise = s % self.n_levels
        sharp = (s // self.n_levels) % self.n_levels
        contrast = s // (self.n_levels ** 2)
        return contrast, sharp, noise
    
    def features_to_state(self, contrast, sharp, noise):
        c = np.clip(contrast, 0, self.n_levels - 1)
        s = np.clip(sharp, 0, self.n_levels - 1)
        n = np.clip(noise, 0, self.n_levels - 1)
        return int(c * self.n_levels ** 2 + s * self.n_levels + n)
    
    def _build_model(self):
        S, A = self.n_states, self.n_actions
        self.P = np.zeros((S, A, S))
        self.R = np.zeros((S, A))
        
        # Terminal state: self-loop with zero reward
        for a in range(A):
            self.P[self.terminal_state, a, self.terminal_state] = 1.0
        
        for s in range(S - 1):
            c, sh, n = self.state_to_features(s)
            
            # 'Done' action → terminal
            quality = (c + sh + (self.n_levels - 1 - n)) / (3 * (self.n_levels - 1))
            self.P[s, 5, self.terminal_state] = 1.0
            self.R[s, 5] = quality * 5  # Final quality bonus
            
            # Sharpen: increases sharpness, might increase noise
            new_sh = min(sh + 1, self.n_levels - 1)
            new_n = min(n + 1, self.n_levels - 1) if np.random.rand() > 0.5 else n
            s_next = self.features_to_state(c, new_sh, n)
            s_next_noisy = self.features_to_state(c, new_sh, new_n)
            self.P[s, 0, s_next] = 0.6
            self.P[s, 0, s_next_noisy] += 0.4
            self.R[s, 0] = 0.3 if sh < self.n_levels - 1 else -0.1
            
            # Denoise: reduces noise, might reduce sharpness
            new_n2 = max(n - 1, 0)
            new_sh2 = max(sh - 1, 0)
            s_next = self.features_to_state(c, sh, new_n2)
            s_next_blur = self.features_to_state(c, new_sh2, new_n2)
            self.P[s, 1, s_next] = 0.7
            self.P[s, 1, s_next_blur] += 0.3
            self.R[s, 1] = 0.2 if n > 0 else -0.1
            
            # Contrast+: increases contrast
            new_c = min(c + 1, self.n_levels - 1)
            s_next = self.features_to_state(new_c, sh, n)
            self.P[s, 2, s_next] = 1.0
            self.R[s, 2] = 0.3 if c < self.n_levels - 1 else -0.2
            
            # Histogram Eq: normalizes contrast to medium-high, reduces noise slightly
            target_c = min(c + 1, self.n_levels - 1)
            s_next = self.features_to_state(target_c, sh, max(n - 1, 0))
            self.P[s, 3, s_next] = 0.8
            self.P[s, 3, self.features_to_state(target_c, sh, n)] += 0.2
            self.R[s, 3] = 0.2
            
            # Edge Enhance: increases sharpness a lot but adds noise
            new_sh3 = min(sh + 2, self.n_levels - 1)
            new_n3 = min(n + 1, self.n_levels - 1)
            s_next = self.features_to_state(c, new_sh3, new_n3)
            self.P[s, 4, s_next] = 1.0
            self.R[s, 4] = 0.1
            
            self.R[s, :] -= 0.05  # Small step cost

img_mdp = ImageProcessingMDP(n_levels=5)
print(f"Image Processing MDP: {img_mdp.n_states} states, {img_mdp.n_actions} actions")
print(f"Actions: {img_mdp.action_names}")

In [ ]:
# Run value iteration on image processing MDP
V_img, pi_img, _, deltas_img = value_iteration(img_mdp, gamma=0.95, theta=1e-6)

# Visualize optimal policy for different starting states
print("\nOptimal Policy for Selected States:")
print("=" * 65)
print(f"{'State (C, S, N)':>20} | {'Best Action':>15} | {'Value':>8}")
print("-" * 65)

for c in range(img_mdp.n_levels):
    for sh in [0, 2, 4]:
        for n in [0, 2, 4]:
            s = img_mdp.features_to_state(c, sh, n)
            best_a = np.argmax(pi_img[s])
            print(f"  ({c}, {sh}, {n}){' ':>12} | {img_mdp.action_names[best_a]:>15} | {V_img[s]:>8.2f}")

# Simulate the optimal pipeline from a degraded state
print("\n" + "=" * 65)
print("Optimal Pipeline from Degraded Image (low contrast, low sharp, high noise):")
print("=" * 65)

s = img_mdp.features_to_state(0, 0, 4)  # Worst state
total_reward = 0
step = 0

while s != img_mdp.terminal_state and step < 20:
    features = img_mdp.state_to_features(s)
    best_a = np.argmax(pi_img[s])
    print(f"  Step {step}: State (C={features[0]}, S={features[1]}, N={features[2]}) "
          f"→ Action: {img_mdp.action_names[best_a]}")
    
    # Take the most likely transition
    s = np.argmax(img_mdp.P[s, best_a])
    total_reward += img_mdp.R[s, best_a] if s != img_mdp.terminal_state else 0
    step += 1

print(f"  → Reached terminal state after {step} steps")

In [ ]:
# Visualize value function across image quality dimensions
fig, axes = plt.subplots(1, img_mdp.n_levels, figsize=(4 * img_mdp.n_levels, 4))

for noise_level in range(img_mdp.n_levels):
    grid = np.zeros((img_mdp.n_levels, img_mdp.n_levels))
    for c in range(img_mdp.n_levels):
        for s in range(img_mdp.n_levels):
            state = img_mdp.features_to_state(c, s, noise_level)
            grid[c, s] = V_img[state]
    
    im = axes[noise_level].imshow(grid, cmap='RdYlGn', interpolation='nearest',
                                  origin='lower')
    axes[noise_level].set_xlabel('Sharpness')
    if noise_level == 0:
        axes[noise_level].set_ylabel('Contrast')
    axes[noise_level].set_title(f'Noise = {noise_level}')
    plt.colorbar(im, ax=axes[noise_level], shrink=0.8)

plt.suptitle('Optimal Value Function V*(contrast, sharpness) for Different Noise Levels',
             fontsize=13)
plt.tight_layout()
plt.show()

---

## 📝 Summary

In this notebook, we covered:

1. **Bellman Equations** — the recursive structure of value functions ($v_\pi$, $q_\pi$, $v_*$, $q_*$)
2. **Policy Evaluation** — iterative computation of $v_\pi$ via Bellman expectation updates; convergence guaranteed by contraction mapping
3. **Policy Improvement** — constructing a better policy by acting greedily w.r.t. $v_\pi$
4. **Policy Iteration** — alternating evaluation and improvement; converges to $\pi_*$ in finite steps
5. **Value Iteration** — combining evaluation and improvement into one update; converges to $v_*$
6. **Image Processing MDP** — DP applied to finding the optimal sequence of image enhancement operations

### 🔑 Key Equations

| Concept | Formula |
|---------|----------|
| Bellman Expectation | $v_\pi(s) = \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a)[r + \gamma v_\pi(s')]$ |
| Bellman Optimality | $v_*(s) = \max_a \sum_{s',r} p(s',r|s,a)[r + \gamma v_*(s')]$ |
| Contraction | $\|T_\pi v - T_\pi u\|_\infty \leq \gamma \|v - u\|_\infty$ |
| Matrix Form | $\mathbf{v}_\pi = (\mathbf{I} - \gamma \mathbf{P}_\pi)^{-1} \mathbf{r}_\pi$ |

---

## ➡️ Next

In the next notebook, we'll explore **Monte Carlo Methods** — learning value functions from sampled episodes without needing a model of the environment.